In [21]:
import numpy as np
import math
from scipy import stats

In [14]:
#part 1
n = 100

U = np.random.uniform(size=n)
X = np.exp(U)

mean = X.mean()
s = X.std(ddof=1)
se = s / np.sqrt(n)

z = 1.96
ci_low  = mean - z * se
ci_high = mean + z * se


print(mean)
print(s)
print(se)
print(ci_low, ci_high)

1.7256083189307279
0.5249917758742838
0.05249917758742838
1.6227099308593682 1.8285067070020875


In [9]:
# part 2
n = int(n / 2)

U = np.random.uniform(size=n)
X = (np.exp(U) + np.exp(1 - U)) / 2

mean = X.mean()
s = X.std(ddof=1)
se = s / np.sqrt(n)

ci_low  = mean - z * se
ci_high = mean + z * se

print(mean)
print(s)
print(se)
print(ci_low, ci_high)

1.739191023683069
0.05887162810172546
0.016994775166081364
1.7058812643575496 1.7725007830085884


In [5]:
# part 3
n = 100
U = np.random.uniform(size=n)
X = np.exp(U)
Z = U
c = -np.cov(X, Z)[0, 1] / np.var(Z, ddof=1)
Y = X + c * (Z - 0.5)
mean = Y.mean()
s = Y.std(ddof=1)
se = s / np.sqrt(n)
ci_low  = mean - z * se
ci_high = mean + z * se

print(mean)
print(s)
print(se)
print(ci_low, ci_high)

1.721527435321104
0.06509946152058842
0.006509946152058842
1.7087679408630687 1.7342869297791395


In [6]:
# part 4
k = 20
m = 5
n = k * m
strata = np.arange(k).reshape(-1, 1)
U = (strata + np.random.uniform(size=(k, m))) / k
X = np.exp(U)
mean = X.mean()
se = np.sqrt(X.var(axis=1, ddof=1).sum() / (k**2 * m))
ci_low  = mean - z * se
ci_high = mean + z * se

print(mean)
print(se)
print(ci_low, ci_high)

1.7139348300409858
0.002654456974547825
1.7087320943708721 1.7191375657110994


In [ ]:
# part 5
class BlockingSystem():
    def __init__(self, arrival_sampler, service_mean=8, num_servers=10, num_customers=10*10000):
        self.arrival_sampler = arrival_sampler
        self.service_mean = service_mean
        self.num_servers = num_servers
        self.num_customers = num_customers
        self.time = 0
        self.servers = [(0, 0)] * num_servers # (start time, service time)
        self.blocked = 0
        self.sum_service_times = 0

    def sample_customer(self):
        arrival_delta = self.arrival_sampler()
        service_time = np.random.exponential(self.service_mean)
        
        self.sum_service_times += service_time
        self.time += arrival_delta

        return arrival_delta, service_time, self.time 

    def start_processing(self, customer, server):
        arrival_delta, service_time, arrived_timestamp = customer
        start_time = arrived_timestamp
        
        self.servers[server] = [start_time, service_time]

    def get_first_available_server(self):
        for i in range(len(self.servers)):
            if (self.servers[i][0] + self.servers[i][1] <= self.time):
                return i
        
        return -1

    
    def simulate(self):
        while self.num_customers > 0:
            customer = self.sample_customer()
            server = self.get_first_available_server()

            if (server == -1):
                self.blocked += 1
            else:
                self.start_processing(customer, server)

            self.num_customers -= 1

        return self.blocked, self.sum_service_times

def erlang_b_formula(arrival_mean, service_mean, m):
    A = arrival_mean * service_mean
    den = 0
    for i in range(m+1):
        den += (A**i / math.factorial(i))

    return ((A**m)/math.factorial(m))/den
        
# We conduct 100 independent blocking system experiments. 
# For each experiment, X[i] represents our estimate of the blocking rate, 
# and Z[i] represents the total service time (control variable). 
# E(Z) = mu_Z = num_customers*service_mean.

# Service times that are too long keep the servers busy for a longer period, 
# increasing the number of blockages so there is a positive correlation between X and Z. 
# This property allows us to decrease the variance through corrected observations,
# Y[i] = X[i] + c*(Z[i]-mu_Z), where the coefficient c is selected to minimize the variance Var(Y).

n_runs        = 100
num_customers = 10000
arrival_mean  = 1
service_mean  = 8
mu_Z          = num_customers * service_mean  # media nota di Z

X = np.zeros(n_runs)   # blocking rate per run
Z = np.zeros(n_runs)   # sum service times per run

for i in range(n_runs):
    bs = BlockingSystem(
        lambda: np.random.exponential(arrival_mean),
        service_mean, 10, num_customers
    )
    blocked, sum_st = bs.simulate()
    X[i] = blocked / num_customers
    Z[i] = sum_st

# optimal coeff 
c = -np.cov(X, Z)[0, 1] / np.var(Z, ddof=1)

Y = X + c * (Z - mu_Z)
z_val = 1.96

# crude MC
mean_mc = X.mean()
se_mc   = X.std(ddof=1) / np.sqrt(n_runs)

# using CV
mean_cv = Y.mean()
se_cv   = Y.std(ddof=1) / np.sqrt(n_runs)

print(f"Crude MC: {mean_mc}  CI: [{mean_mc - z_val*se_mc}, {mean_mc + z_val*se_mc}]")
print(f"CV:       {mean_cv}  CI: [{mean_cv - z_val*se_cv}, {mean_cv + z_val*se_cv}]")
print(f"Var MC:   {X.var(ddof=1)}")
print(f"Var CV:   {Y.var(ddof=1)}")
print(f"c: {c}")

Crude MC: 0.12189  CI: [0.12074, 0.12305]
CV:       0.12141  CI: [0.12047, 0.12236]
Var MC:   0.000035
Var CV:   0.000023
c: -4.198045077353676e-06


In [ ]:
# part 7

# we estimate P(Z > a) for Z = N(0,1) using two methods:
# - Crude MC: sample from N(0,1) and count how many exceed a.
#   For large a this is very inefficient because the event is rare.
# - IS: sample from g = N(a,1), centered where the event actually happens.
#   Each sample is reweighted by f(Y)/g(Y) to correct for sampling
#   from the wrong distribution. Same answer, much lower variance.

def estimate_tail_prob(a, n, sigma2=1):
    sigma = np.sqrt(sigma2)
    
    # MC
    Z = np.random.normal(0, 1, size=n)
    mc_estimate = np.mean(Z > a)
    mc_var      = np.var(Z > a, ddof=1)

    # importance sampling
    Y = np.random.normal(a, sigma, size=n)
    f = stats.norm.pdf(Y, 0, 1)      # N(0,1) 
    g = stats.norm.pdf(Y, a, sigma)  # N(a,sigma) 
    weights = f / g
    is_estimate = np.mean((Y > a) * weights)
    is_var      = np.var((Y > a) * weights, ddof=1)

    print(f"a={a}, n={n}")
    print(f"True value : {1 - stats.norm.cdf(a)}")
    print(f"Crude MC   : {mc_estimate}  Var: {mc_var}")
    print(f"IS         : {is_estimate}  Var: {is_var}\n")

for a in [2, 4]:
    for n in [1000, 10000]:
        estimate_tail_prob(a, n)

a=2, n=1000
True value : 0.022750
Crude MC   : 0.022  Var: 0.02153753753753754
IS         : 0.024692990127427143  Var: 0.0012573555535143695

a=2, n=10000
True value : 0.022750
Crude MC   : 0.0222  Var: 0.021709330933093317
IS         : 0.02276737045584175  Var: 0.001192538026589263

a=4, n=1000
True value : 0.000032
Crude MC   : 0.0  Var: 0.0
IS         : 3.082259168422843e-05  Var: 4.397840931140415e-09

a=4, n=10000
True value : 0.000032
Crude MC   : 0.0001  Var: 0.00010000000000000005
IS         : 3.147471468305224e-05  Var: 4.471763843407211e-09



In [ ]:
# part 8

def sample_exp(lam, n):
    U = np.random.uniform(size=n)
    Y = -np.log(1 - U * (1 - np.exp(-lam))) / lam
    
    g = lam * np.exp(-lam * Y) / (1 - np.exp(-lam))
    weights = np.exp(Y) / g
    
    return weights.mean(), weights.var(ddof=1)

n = 10000

for lam in [0.5, 1.0, 2.0, 5.0]:
    est, var = sample_exp(lam, n)
    print(f"lambda={lam}  estimate={est}  Var={var}")

print()
print(np.e - 1)

lambda=0.5  estimate=1.719197  Var=0.566705
lambda=1.0  estimate=1.734320  Var=1.091739
lambda=2.0  estimate=1.721409  Var=2.885193
lambda=5.0  estimate=1.695526  Var=28.469250

1.718281828459045


In [ ]:
# part 9

# Pareto has density f(x) = k/beta * (x/beta)^(-k-1) and the first moment distribution has density g(x) = (k-1)/β * (x/β)^(-k)

# The IS weight is f(x) / g(x) = k/(k-1) * beta/x

# So the IS estimate of the mean is:
# E_g[ x * f(x)/g(x) ] = E_g[ x * k/(k-1) * beta/x ]
#                         = E_g[ k*beta/(k-1) ]
#                        = k*beta/(k-1)

#  Which is formula for the Pareto mean => the variance is zero